# YOLO11s Training for BDD100K Road-Object Detection

This notebook trains and evaluates YOLO11s for six road-object classes: car, bus, truck, pedestrian, traffic light, and traffic sign. The workflow combines class-balanced sampling, transfer learning, a higher-resolution refinement stage, held-out testing, confidence calibration, and a real-time benchmark.

The selected model achieved 67.11% test precision, 55.06% recall, 60.49% F1, 59.67% mAP50, and 33.41% mAP50:95 on 2,000 held-out images. Object detection is reported with precision, recall, F1, and mAP because confidence is not the same as accuracy.

## Dataset And Training Strategy

BDD100K contains many cars but fewer buses, trucks, pedestrians, lights, and signs. Class-aware sampling ensures that rare classes appear often enough during training. The main stage uses 4,800 images at 576 px for broad scene coverage, while refinement uses 3,200 more strongly balanced images at 704 px to preserve small-object detail.

Transfer learning begins with learned YOLO11s visual features instead of learning every feature from scratch. This improves convergence and reduces the amount of task-specific training required.

In [9]:
from __future__ import annotations

import gc
import json
import math
import os
import random
import shutil
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from road_detection.yolo_training_utils import (
    label_path_for_image,
    prepare_fast_data_files,
    read_image_manifest,
    stage_fast_data_files,
    stage_images_and_labels,
    uniform_sample,
    write_image_manifest,
)

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Ultralytics:", __import__("ultralytics").__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

EXPECTED_VENV = Path(r"C:\tf214_hw2\Scripts\python.exe")
if EXPECTED_VENV.exists() and Path(sys.executable).resolve() != EXPECTED_VENV.resolve():
    raise RuntimeError(
        "Select the Jupyter kernel 'Python (tf214_hw2)' and restart the notebook. "
        f"Current interpreter: {sys.executable}"
    )

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

Python: c:\tf214_hw2\Scripts\python.exe
PyTorch: 2.10.0+cu126
Ultralytics: 8.4.112
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [10]:
# Use "smoke" once after changing code. Use "rtx3050_more_data" for the real run.
RUN_MODE = "rtx3050_more_data"
BASE_RUN_TAG = "yolo11s_576_class_coverage_v4_56epochs"
RESUME_TRAINING = False
RUN_REFINEMENT = True

PROFILES = {
    "smoke": {
        "model": "yolo11s.pt",
        "main_count": 128,
        "refine_count": 128,
        "epoch_val_count": 64,
        "candidate_scan_count": 512,
        "main_epochs": 2,
        "refine_epochs": 1,
        "imgsz": 512,
        "refine_imgsz": 576,
        "batch": 8,
        "refine_batch": 8,
        "patience": 3,
        "final_eval_count": 128,
        "main_class_minimums": [100, 20, 30, 40, 60, 90],
        "refine_class_minimums": [100, 25, 35, 45, 70, 95],
    },
    "rtx3050_more_data": {
        "model": "yolo11s.pt",
        "main_count": 4800,
        "refine_count": 3200,
        "epoch_val_count": 600,
        "candidate_scan_count": 40000,
        "main_epochs": 56,
        "refine_epochs": 14,
        "imgsz": 576,
        "refine_imgsz": 704,
        "batch": 16,
        "refine_batch": 12,
        "patience": 8,
        "final_eval_count": 2000,
        "main_class_minimums": [4500, 1000, 1500, 1800, 2800, 4000],
        "refine_class_minimums": [3000, 800, 1100, 1300, 2000, 2800],
    },
    "overnight_accuracy": {
        "model": "yolo11s.pt",
        "main_count": 6400,
        "refine_count": 4800,
        "epoch_val_count": 800,
        "candidate_scan_count": 60000,
        "main_epochs": 63,
        "refine_epochs": 12,
        "imgsz": 576,
        "refine_imgsz": 704,
        "batch": 16,
        "refine_batch": 12,
        "patience": 10,
        "final_eval_count": 2000,
        "main_class_minimums": [6000, 1400, 2200, 2600, 3800, 5400],
        "refine_class_minimums": [4500, 1200, 1700, 2000, 3000, 4200],
    },
}
CFG = PROFILES[RUN_MODE]
RUN_TAG = f"{BASE_RUN_TAG}_{RUN_MODE}"

if RUN_MODE != "smoke" and not torch.cuda.is_available():
    raise RuntimeError(
        "The real profiles require CUDA. The prior notebook accidentally used "
        "CPU-only PyTorch; repair/select the tf214_hw2 kernel before training."
    )

DEVICE = 0 if torch.cuda.is_available() else "cpu"
# RAM-cached datasets are fastest with no Windows worker respawn/copy overhead.
WORKERS = 0
DATA_YAML = PROJECT_ROOT / "data" / "bdd100k_yolo" / "data.yaml"
RUN_ROOT = PROJECT_ROOT / "runs" / "notebooks" / "yolo_accuracy"
MANIFEST_DIR = RUN_ROOT / "manifests" / RUN_TAG
PRIOR_BEST = PROJECT_ROOT / "outputs" / "yolo_final" / "bdd100k_yolo11s_best.pt"
START_MODEL = str(PRIOR_BEST) if PRIOR_BEST.exists() else CFG["model"]
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "yolo_more_data"
FAST_CACHE_BASE = Path(
    os.environ.get("BDD100K_FAST_CACHE")
    or Path(os.environ.get("LOCALAPPDATA", PROJECT_ROOT / "tmp"))
    / "bdd100k_road_detection_cache"
)
LOCAL_CACHE_ROOT = FAST_CACHE_BASE / RUN_TAG
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_YAML.exists():
    raise FileNotFoundError(
        f"Missing {DATA_YAML}. Run scripts/prepare_bdd100k.ps1 with the real "
        "BDD100K folder first."
    )
print(pd.Series(CFG, name=RUN_MODE))

model                                              yolo11s.pt
main_count                                               4800
refine_count                                             3200
epoch_val_count                                           600
candidate_scan_count                                    40000
main_epochs                                                56
refine_epochs                                              14
imgsz                                                     576
refine_imgsz                                              704
batch                                                      16
refine_batch                                               12
patience                                                    8
final_eval_count                                         2000
main_class_minimums      [4500, 1000, 1500, 1800, 2800, 4000]
refine_class_minimums     [3000, 800, 1100, 1300, 2000, 2800]
Name: rtx3050_more_data, dtype: object


In [11]:
data_files = prepare_fast_data_files(
    data_yaml=DATA_YAML,
    output_dir=MANIFEST_DIR,
    run_root=RUN_ROOT,
    preferred_manifest_tag=RUN_TAG,
    main_count=CFG["main_count"],
    refine_count=CFG["refine_count"],
    validation_count=CFG["epoch_val_count"],
    seed=SEED,
    candidate_scan_count=CFG["candidate_scan_count"],
    scan_workers=12,
    main_class_minimums=CFG["main_class_minimums"],
    refine_class_minimums=CFG["refine_class_minimums"],
)
staging_started = time.perf_counter()
data_files = stage_fast_data_files(
    data_files,
    cache_root=LOCAL_CACHE_ROOT,
    source_data_yaml=DATA_YAML,
    workers=12,
)
print(f"One-time/reusable local staging: {time.perf_counter() - staging_started:.1f}s")
print("Sampling source:", data_files.source)
print("Main images:", sum(1 for _ in data_files.main_manifest.open()))
print("Refine images:", sum(1 for _ in data_files.refine_manifest.open()))
print("Epoch validation images:", sum(1 for _ in data_files.validation_manifest.open()))
print("Main YAML:", data_files.main_yaml)


def class_coverage_table(manifest):
    image_counts = np.zeros(6, dtype=int)
    box_counts = np.zeros(6, dtype=int)
    images = read_image_manifest(manifest)
    for image_path in images:
        seen = set()
        label_path = label_path_for_image(image_path)
        if not label_path.exists():
            continue
        for line in label_path.read_text(encoding="utf-8").splitlines():
            fields = line.split()
            if len(fields) < 5:
                continue
            class_id = int(float(fields[0]))
            box_counts[class_id] += 1
            seen.add(class_id)
        for class_id in seen:
            image_counts[class_id] += 1
    return pd.DataFrame({
        "class": ["car", "bus", "truck", "pedestrian", "traffic light", "traffic sign"],
        "images": image_counts,
        "boxes": box_counts,
        "image_percent": 100.0 * image_counts / len(images),
    })

print("Main class coverage")
display(class_coverage_table(data_files.main_manifest))
print("Refinement class coverage")
display(class_coverage_table(data_files.refine_manifest))

One-time/reusable local staging: 43.9s
Sampling source: new small-object-aware candidate index C:\Users\vipra\OneDrive\Documents\GitHub\ECGR-5106-Intro-To-Deep-Learning\Final_Project\runs\notebooks\yolo_accuracy\manifests\yolo11s_576_class_coverage_v4_56epochs_rtx3050_more_data\candidate_index.jsonl (40000 scenes); staged in local cache C:\Users\vipra\AppData\Local\bdd100k_road_detection_cache\yolo11s_576_class_coverage_v4_56epochs_rtx3050_more_data
Main images: 4800
Refine images: 3200
Epoch validation images: 600
Main YAML: C:\Users\vipra\AppData\Local\bdd100k_road_detection_cache\yolo11s_576_class_coverage_v4_56epochs_rtx3050_more_data\manifests\main.yaml
Main class coverage


,class,images,boxes,image_percent
0,car,4756,48838,99.083333
1,bus,1760,2270,36.666667
2,truck,2329,3679,48.520833
3,pedestrian,2587,11693,53.895833
4,traffic light,3527,17657,73.479167
5,traffic sign,4311,19539,89.812500


Refinement class coverage


,class,images,boxes,image_percent
0,car,3176,33140,99.25000
1,bus,1415,1842,44.21875
2,truck,1706,2782,53.31250
3,pedestrian,1877,8866,58.65625
4,traffic light,2466,12608,77.06250
5,traffic sign,2903,13388,90.71875


## Stage 1: Main Transfer Learning

The main stage trains for 56 continuous epochs using 4,800 images per epoch, batch size 16, AdamW, AMP, and cosine learning-rate decay from `0.0008`. Rectangular batches reduce unnecessary image padding, while mild color, scale, translation, rotation, and horizontal-flip augmentation improve road-scene variety.

The box, classification, and DFL losses decreased from 1.491, 1.362, and 0.998 to 1.254, 0.906, and 0.911. This indicates that the model continued learning object classes and bounding-box locations throughout the main stage.

In [12]:
MAIN_NAME = f"{RUN_TAG}_main"
MAIN_DIR = RUN_ROOT / MAIN_NAME
MAIN_LAST = MAIN_DIR / "weights" / "last.pt"

if RESUME_TRAINING and MAIN_LAST.exists():
    print("Resuming:", MAIN_LAST)
    main_model = YOLO(str(MAIN_LAST))
    main_model.train(resume=True)
else:
    print("Starting from:", START_MODEL)
    main_model = YOLO(START_MODEL)
    main_model.train(
        data=str(data_files.main_yaml),
        epochs=CFG["main_epochs"],
        imgsz=CFG["imgsz"],
        batch=CFG["batch"],
        device=DEVICE,
        workers=WORKERS,
        cache="ram" if torch.cuda.is_available() else False,
        rect=True,
        amp=torch.cuda.is_available(),
        optimizer="AdamW",
        lr0=0.0008,
        lrf=0.05,
        cos_lr=True,
        warmup_epochs=1.0,
        weight_decay=0.0005,
        box=7.5,
        cls=0.85,
        cls_pw=0.25,
        dfl=1.5,
        hsv_h=0.015,
        hsv_s=0.55,
        hsv_v=0.35,
        degrees=2.0,
        translate=0.08,
        scale=0.25,
        shear=0.0,
        perspective=0.0002,
        fliplr=0.5,
        flipud=0.0,
        mosaic=0.0,
        mixup=0.0,
        close_mosaic=0,
        # Fixed shapes avoid repeated cuDNN autotuning on this 4 GB GPU.
        patience=CFG["patience"],
        max_det=300,
        plots=True,
        deterministic=False,
        seed=SEED,
        project=str(RUN_ROOT),
        name=MAIN_NAME,
        exist_ok=True,
        verbose=True,
    )

MAIN_BEST = MAIN_DIR / "weights" / "best.pt"
if not MAIN_BEST.exists():
    raise FileNotFoundError(f"Training did not create {MAIN_BEST}")
print("Main candidate:", MAIN_BEST)
del main_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Starting from: C:\Users\vipra\OneDrive\Documents\GitHub\ECGR-5106-Intro-To-Deep-Learning\Final_Project\outputs\yolo_final\bdd100k_yolo11s_best.pt
New https://pypi.org/project/ultralytics/8.4.115 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.112  Python-3.11.9 torch-2.10.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=False, classes=None, close_mosaic=0, cls=0.85, cls_pw=0.25, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\vipra\AppData\Local\bdd100k_road_detection_cache\yolo11s_576_class_coverage_v4_56epochs_rtx3050_more_data\manifests\main.yaml, degrees=2.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, e

## Stage 2: Small-Object Refinement

Refinement starts from the best main-stage checkpoint and trains at 704 px using 3,200 class-balanced images, batch size 12, and a lower `0.0003` learning rate. The larger input retains more information for pedestrians, traffic lights, and traffic signs.

Mosaic and mixup are disabled so small objects are not reduced further. Early stopping ended refinement after 12 epochs, and epoch 5 produced the best validation checkpoint.

In [13]:
REFINE_NAME = f"{RUN_TAG}_refine"
REFINE_DIR = RUN_ROOT / REFINE_NAME
REFINE_LAST = REFINE_DIR / "weights" / "last.pt"
REFINE_BEST = REFINE_DIR / "weights" / "best.pt"

if RUN_REFINEMENT:
    if RESUME_TRAINING and REFINE_LAST.exists():
        print("Resuming:", REFINE_LAST)
        refine_model = YOLO(str(REFINE_LAST))
        refine_model.train(resume=True)
    else:
        refine_model = YOLO(str(MAIN_BEST))
        refine_model.train(
            data=str(data_files.refine_yaml),
            epochs=CFG["refine_epochs"],
            imgsz=CFG["refine_imgsz"],
            batch=CFG["refine_batch"],
            device=DEVICE,
            workers=WORKERS,
            cache="ram" if torch.cuda.is_available() else False,
            rect=True,
            amp=torch.cuda.is_available(),
            optimizer="AdamW",
            lr0=0.00030,
            lrf=0.10,
            cos_lr=True,
            warmup_epochs=1.0,
            weight_decay=0.0005,
            box=8.0,
            cls=0.90,
            cls_pw=0.35,
            dfl=1.5,
            hsv_h=0.012,
            hsv_s=0.40,
            hsv_v=0.28,
            degrees=1.0,
            translate=0.05,
            scale=0.20,
            fliplr=0.5,
            mosaic=0.0,
            mixup=0.0,
            close_mosaic=0,
            patience=max(4, CFG["refine_epochs"] // 2),
            max_det=300,
            plots=True,
            deterministic=False,
            seed=SEED + 1,
            project=str(RUN_ROOT),
            name=REFINE_NAME,
            exist_ok=True,
            verbose=True,
        )
    if not REFINE_BEST.exists():
        raise FileNotFoundError(f"Refinement did not create {REFINE_BEST}")
    del refine_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

New https://pypi.org/project/ultralytics/8.4.115 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.112  Python-3.11.9 torch-2.10.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=12, bgr=0.0, box=8.0, cache=ram, cfg=None, channels_last=False, classes=None, close_mosaic=0, cls=0.9, cls_pw=0.35, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\vipra\AppData\Local\bdd100k_road_detection_cache\yolo11s_576_class_coverage_v4_56epochs_rtx3050_more_data\manifests\refine.yaml, degrees=1.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=14, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.012, hsv_s=0.4, hsv_v=0.28, 

## Validation-Based Model Selection

Available checkpoints are evaluated on the same fixed 600-image validation subset. The selection score combines 55% mAP50, 30% F1, and 15% mAP50:95, rewarding correct detections, balanced precision and recall, and accurate box placement.

The highest-scoring checkpoint is copied as the deployment model. It is then evaluated once on separate 2,000-image validation and test subsets.

In [14]:
def summarize_yolo(metrics):
    precision = float(metrics.box.mp)
    recall = float(metrics.box.mr)
    f1 = 2 * precision * recall / max(1e-12, precision + recall)
    map50 = float(metrics.box.map50)
    map50_95 = float(metrics.box.map)
    quality = 0.55 * map50 + 0.30 * f1 + 0.15 * map50_95
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "map50": map50,
        "map50_95": map50_95,
        "quality": quality,
    }

candidates = {"main": MAIN_BEST}
if RUN_REFINEMENT and REFINE_BEST.exists():
    candidates["refine"] = REFINE_BEST
if PRIOR_BEST.exists():
    candidates["completed_01_baseline"] = PRIOR_BEST
OLD_BASELINE = (
    PROJECT_ROOT / "runs" / "notebooks" / "yolo"
    / "bdd100k_cpu_quick_finetuned" / "weights" / "best.pt"
)
if OLD_BASELINE.exists():
    candidates["old_cpu_baseline"] = OLD_BASELINE

candidate_rows = []
candidate_metrics = {}
for name, weights in candidates.items():
    print(f"Evaluating {name}: {weights}")
    metrics = YOLO(str(weights)).val(
        data=str(data_files.main_yaml),
        imgsz=CFG["refine_imgsz"],
        batch=CFG["refine_batch"],
        device=DEVICE,
        workers=WORKERS,
        conf=0.001,
        iou=0.70,
        max_det=300,
        plots=False,
        verbose=False,
    )
    row = {"candidate": name, "weights": str(weights), **summarize_yolo(metrics)}
    candidate_rows.append(row)
    candidate_metrics[name] = metrics

candidate_table = pd.DataFrame(candidate_rows).sort_values("quality", ascending=False)
display(candidate_table)
winner_name = str(candidate_table.iloc[0]["candidate"])
winner_path = Path(candidate_table.iloc[0]["weights"])
winner_metrics = candidate_metrics[winner_name]
FINAL_WEIGHTS = OUTPUT_DIR / "bdd100k_yolo11s_more_data_best.pt"
shutil.copy2(winner_path, FINAL_WEIGHTS)
candidate_table.to_csv(OUTPUT_DIR / "candidate_comparison.csv", index=False)
print("Selected:", winner_name)
print("Deployment weights:", FINAL_WEIGHTS)

Evaluating main: C:\Users\vipra\OneDrive\Documents\GitHub\ECGR-5106-Intro-To-Deep-Learning\Final_Project\runs\notebooks\yolo_accuracy\yolo11s_576_class_coverage_v4_56epochs_rtx3050_more_data_main\weights\best.pt
Ultralytics 8.4.112  Python-3.11.9 torch-2.10.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLO11s summary (fused): 101 layers, 9,415,122 parameters, 0 gradients, 21.3 GFLOPs
WARNING val: Slow image access detected (ping: 0.50.2 ms, read: 47.518.9 MB/s, size: 63.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning C:\Users\vipra\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\Local\bdd100k_road_detection_cache\yolo11s_576_class_coverage_v4_56epochs_rtx3050_more_data\labels\val.cache... 600 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 600/600  0.0s
                 Class     Images  Instances      Box(P          R   

,candidate,weights,precision,recall,f1,map50,map50_95,quality
1,refine,C:\Users\vipra\OneDrive\Documents\GitHub\ECGR-...,0.639269,0.530930,0.580084,0.575870,0.320284,0.538796
0,main,C:\Users\vipra\OneDrive\Documents\GitHub\ECGR-...,0.629293,0.525914,0.572978,0.552806,0.296533,0.520416
2,completed_01_baseline,C:\Users\vipra\OneDrive\Documents\GitHub\ECGR-...,0.592654,0.512522,0.549683,0.518241,0.279578,0.491874
3,old_cpu_baseline,C:\Users\vipra\OneDrive\Documents\GitHub\ECGR-...,0.479694,0.369609,0.417517,0.350846,0.182470,0.345591


Selected: refine
Deployment weights: C:\Users\vipra\OneDrive\Documents\GitHub\ECGR-5106-Intro-To-Deep-Learning\Final_Project\outputs\yolo_more_data\bdd100k_yolo11s_more_data_best.pt


In [15]:
# Build reproducible, bounded final validation and test manifests.
base_config = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
dataset_root = Path(base_config["path"])
if not dataset_root.is_absolute():
    dataset_root = (DATA_YAML.parent / dataset_root).resolve()

def split_dir(split_name):
    value = Path(base_config[split_name])
    return value if value.is_absolute() else dataset_root / value

final_val_images = uniform_sample(
    sorted(split_dir("val").glob("*.jpg")),
    CFG["final_eval_count"],
    SEED + 20,
)
final_test_dir = split_dir("test") if "test" in base_config else None
if final_test_dir is None or not final_test_dir.exists():
    raise FileNotFoundError("Converted data.yaml must contain the held-out local test split.")
final_test_images = uniform_sample(
    sorted(final_test_dir.glob("*.jpg")),
    CFG["final_eval_count"],
    SEED + 21,
)
final_val_images = stage_images_and_labels(
    final_val_images, LOCAL_CACHE_ROOT, "val", workers=12
)
final_test_images = stage_images_and_labels(
    final_test_images, LOCAL_CACHE_ROOT, "test", workers=12
)
final_val_manifest = write_image_manifest(MANIFEST_DIR / "final_val.txt", final_val_images)
final_test_manifest = write_image_manifest(MANIFEST_DIR / "final_test.txt", final_test_images)

final_config = dict(base_config)
final_config["path"] = dataset_root.as_posix()
final_config["val"] = final_val_manifest.resolve().as_posix()
final_config["test"] = final_test_manifest.resolve().as_posix()
FINAL_DATA_YAML = MANIFEST_DIR / "final_eval.yaml"
FINAL_DATA_YAML.write_text(yaml.safe_dump(final_config, sort_keys=False), encoding="utf-8")

final_model = YOLO(str(FINAL_WEIGHTS))
final_val_metrics = final_model.val(
    data=str(FINAL_DATA_YAML),
    split="val",
    imgsz=CFG["refine_imgsz"],
    batch=CFG["refine_batch"],
    device=DEVICE,
    workers=WORKERS,
    conf=0.001,
    iou=0.70,
    max_det=300,
    plots=True,
    verbose=False,
)
final_test_metrics = final_model.val(
    data=str(FINAL_DATA_YAML),
    split="test",
    imgsz=CFG["refine_imgsz"],
    batch=CFG["refine_batch"],
    device=DEVICE,
    workers=WORKERS,
    conf=0.001,
    iou=0.70,
    max_det=300,
    plots=False,
    verbose=False,
)
final_table = pd.DataFrame(
    [
        {"split": "validation", **summarize_yolo(final_val_metrics)},
        {"split": "test", **summarize_yolo(final_test_metrics)},
    ]
)
display(final_table)
final_table.to_csv(OUTPUT_DIR / "final_metrics.csv", index=False)

Ultralytics 8.4.112  Python-3.11.9 torch-2.10.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLO11s summary (fused): 101 layers, 9,415,122 parameters, 0 gradients, 21.3 GFLOPs
WARNING val: Slow image access detected (ping: 0.30.1 ms, read: 31.564.2 MB/s, size: 59.1 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning C:\Users\vipra\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\Local\bdd100k_road_detection_cache\yolo11s_576_class_coverage_v4_56epochs_rtx3050_more_data\labels\val... 2000 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2000/2000 128.2it/s 15.6s0.1s
val: New cache created: C:\Users\vipra\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\Local\bdd100k_road_detection_cache\yolo11s_576_class_coverage_v4_56epochs_rtx3050_more_data\labels\val.cache
                 Class     Images  In

,split,precision,recall,f1,map50,map50_95,quality
0,validation,0.654717,0.540262,0.592008,0.580777,0.323875,0.545611
1,test,0.671144,0.550617,0.604936,0.596745,0.334114,0.559807


## Confidence Threshold Calibration

The balanced profile selects the confidence threshold that produces the best validation F1 for each class. It is recommended when both precision and recall matter.

The `high_precision_80` profile requires at least 80% measured validation precision and a displayed confidence of at least 0.70. This removes more uncertain detections, but it also lowers recall, especially for distant pedestrians, traffic lights, and signs.

In [16]:
def calibrate_thresholds(metrics, target_precision=0.80, confidence_floor=0.70):
    box = metrics.box
    px = np.asarray(box.px)
    p_curve = np.asarray(box.p_curve)
    r_curve = np.asarray(box.r_curve)
    f1_curve = np.asarray(box.f1_curve)
    names = metrics.names
    balanced = {}
    high_precision = {}
    rows = []
    for class_id in range(len(names)):
        best_f1_index = int(np.nanargmax(f1_curve[class_id]))
        balanced_threshold = float(px[best_f1_index])
        valid = np.flatnonzero(
            (p_curve[class_id] >= target_precision)
            & (px >= confidence_floor)
        )
        if len(valid):
            strict_index = int(valid[np.nanargmax(r_curve[class_id, valid])])
        else:
            strict_index = int(np.argmin(np.abs(px - confidence_floor)))
        strict_threshold = max(confidence_floor, float(px[strict_index]))
        balanced[str(class_id)] = round(max(0.05, balanced_threshold), 3)
        high_precision[str(class_id)] = round(strict_threshold, 3)
        rows.append(
            {
                "class_id": class_id,
                "class": names[class_id],
                "balanced_threshold": balanced[str(class_id)],
                "balanced_f1": float(f1_curve[class_id, best_f1_index]),
                "strict_threshold": high_precision[str(class_id)],
                "strict_precision": float(p_curve[class_id, strict_index]),
                "strict_recall": float(r_curve[class_id, strict_index]),
                "target_precision_met": bool(p_curve[class_id, strict_index] >= target_precision),
            }
        )
    return balanced, high_precision, pd.DataFrame(rows)

balanced_thresholds, strict_thresholds, threshold_table = calibrate_thresholds(
    final_val_metrics
)
display(threshold_table)
threshold_table.to_csv(OUTPUT_DIR / "threshold_calibration.csv", index=False)

deployment_config = {
    "backend": "yolo",
    "weights": FINAL_WEIGHTS.name,
    "imgsz": CFG["refine_imgsz"],
    "max_det": 300,
    "default_threshold_profile": "balanced",
    "threshold_profiles": {
        "balanced": {
            "inference_confidence": 0.05,
            "class_thresholds": balanced_thresholds,
            "meaning": "Per-class validation F1 optimum",
        },
        "high_precision_80": {
            "inference_confidence": 0.70,
            "class_thresholds": strict_thresholds,
            "target_validation_precision": 0.80,
            "meaning": "At least 0.70 displayed confidence; class target is 0.80 precision where supported",
        },
    },
    "selection": winner_name,
    "validation_metrics": summarize_yolo(final_val_metrics),
    "test_metrics": summarize_yolo(final_test_metrics),
}
DEPLOYMENT_CONFIG = OUTPUT_DIR / "deployment_config.json"
DEPLOYMENT_CONFIG.write_text(json.dumps(deployment_config, indent=2), encoding="utf-8")
print("Deployment config:", DEPLOYMENT_CONFIG)

,class_id,class,balanced_threshold,balanced_f1,strict_threshold,strict_precision,strict_recall,target_precision_met
0,0,car,0.327,0.717586,0.701,0.980725,0.373034,True
1,1,bus,0.327,0.525931,0.703,0.805859,0.327103,True
2,2,truck,0.394,0.542334,0.701,0.824774,0.306730,True
3,3,pedestrian,0.249,0.572389,0.701,0.969465,0.060453,True
4,4,traffic light,0.323,0.612593,0.701,0.960855,0.071479,True
5,5,traffic sign,0.280,0.602033,0.701,0.938454,0.185841,True


Deployment config: C:\Users\vipra\OneDrive\Documents\GitHub\ECGR-5106-Intro-To-Deep-Learning\Final_Project\outputs\yolo_more_data\deployment_config.json


In [17]:
# Reload after validation so the predictor owns ordinary, mutable tensors.
benchmark_model = YOLO(str(FINAL_WEIGHTS))
# Warm GPU, measure end-to-end detector throughput, and show representative output.
benchmark_images = [str(path) for path in final_val_images[:100]]
_ = benchmark_model.predict(
    benchmark_images[:8],
    imgsz=CFG["refine_imgsz"],
    batch=CFG["refine_batch"],
    device=DEVICE,
    quantize=16 if torch.cuda.is_available() else None,
    conf=0.25,
    verbose=False,
)
if torch.cuda.is_available():
    torch.cuda.synchronize()
started = time.perf_counter()
benchmark_results = benchmark_model.predict(
    benchmark_images,
    imgsz=CFG["refine_imgsz"],
    batch=CFG["refine_batch"],
    device=DEVICE,
    quantize=16 if torch.cuda.is_available() else None,
    conf=0.25,
    verbose=False,
)
if torch.cuda.is_available():
    torch.cuda.synchronize()
elapsed = time.perf_counter() - started
print(f"End-to-end throughput: {len(benchmark_images) / elapsed:.2f} images/s")
print(f"Latency: {1000 * elapsed / len(benchmark_images):.1f} ms/image")

figure, axes = plt.subplots(2, 2, figsize=(16, 9))
for axis, result in zip(axes.flat, benchmark_results[:4]):
    axis.imshow(result.plot()[..., ::-1])
    axis.axis("off")
plt.tight_layout()
plt.show()

End-to-end throughput: 0.51 images/s
Latency: 1958.1 ms/image


<Figure size 1600x900 with 4 Axes>

## Real-Time YOLO Detection

```powershell
python -m road_detection.realtime_detect `
  --backend yolo `
  --weights outputs\yolo_more_data\bdd100k_yolo11s_more_data_best.pt `
  --config outputs\yolo_more_data\deployment_config.json `
  --threshold-profile balanced `
  --source 0 `
  --device 0